In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Fritsche_Neto2 AlphaSimPy Tutorial

This notebook converts the provided **BRAID breeding program abstraction** into a tutorial-style **AlphaSimPy** simulation notebook.

The breeding program represents an **elite × elite line breeding pipeline** with:
- biparental crossing,
- early-generation genomic selection,
- progeny row advancement,
- regional and advanced yield testing,
- pre-commercial testing,
- foundation increase for breeder seed production.

**Source**: BRAID abstraction  
**Package**: AlphaSimPy  
**Program horizon**: 7 years


## Assumptions Used in the Translation

The BRAID abstraction is high-level, so several implementation details must be made explicit for AlphaSimPy:

1. **Genome simulation** uses `runMacs` with a generic diploid line-breeding setup.
2. **One additive trait** is simulated, matching the BRAID primary selection trait.
3. **Genomic selection** is represented pragmatically using estimated breeding values derived from phenotype-assisted ranking in early generations, because the BRAID abstraction specifies GS accuracy but not a full training-population design.
4. **Selfing-based advancement** is used to move material through F2:F3, regional, and advanced testing stages.
5. **Variable-size stages** (`precommercial_test`, `foundation_increase`) are implemented as downstream selections from the final advanced testing stage.
6. Where exact progeny-per-cross counts are not specified, values are chosen so that stage sizes approximately match the BRAID population sizes.

These assumptions are stated explicitly so the notebook remains readable and runnable.


## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    self as self_pop,
    setPheno,
    selectInd,
    meanG,
    varG,
    mergePops
)

print("AlphaSimPy BRAID conversion tutorial")
print("Libraries imported successfully.")


## Global Parameters

In [ ]:
# Program-level parameters translated from BRAID
program_name = "Fritsche_Neto2"
n_years = 7

# Genome and trait assumptions
n_chr = 20
n_qtl = 100
founder_size = 600
trait_h2 = 0.3

# Stage sizes from BRAID
n_crosses = 300
n_f2_single = 200
n_progeny_rows = 20000
n_regional = 600
n_ayt1 = 60
n_ayt2 = 15

# Explicit implementation assumptions
f1_per_cross = 1
f2f3_per_cross = max(1, n_progeny_rows // n_crosses)   # approximate rows per cross
n_precommercial = max(5, n_ayt2)
n_foundation = max(5, n_ayt2)

# Error variances / replication assumptions by stage
varE_early = 0.6
varE_regional = 0.40
varE_ayt1 = 0.25
varE_ayt2 = 0.15
varE_precommercial = 0.10

rep_regional = 1
rep_ayt1 = 2
rep_ayt2 = 4
rep_precommercial = 6

print("Program:", program_name)
print("Years:", n_years)
print("Founders:", founder_size)
print("Crosses:", n_crosses)
print("Approx. F2:F3 rows per cross:", f2f3_per_cross)


## Create Founders and Simulation Parameters

In [ ]:
print("Creating founder haplotypes and simulation parameters...")

founder_pop = runMacs(
    nInd=founder_size,
    nChr=n_chr,
    segSites=n_qtl,
    inbred=True,
    species="GENERIC"
)

SP = SimParam(founder_pop)
SP.addTraitA(nQtlPerChr=n_qtl, mean=0.0, var=1.0)
SP.setTrackPed(True)

elite_parents = newPop(founder_pop, simParam=SP)

print(f"Elite parents created: {elite_parents.n_ind} individuals")
print(f"Mean G: {meanG(elite_parents)[0]:.3f}")
print(f"Var G: {varG(elite_parents)[0]:.3f}")


## Helper Functions

These helpers keep the notebook readable while mapping the BRAID workflow into AlphaSimPy operations.


In [ ]:
def safeMeanG(pop):
    if pop is None:
        return np.nan
    return float(meanG(pop)[0])

def safeVarG(pop):
    if pop is None:
        return np.nan
    return float(varG(pop)[0])

def stageSummary(stage_name, pop):
    if pop is None:
        return {"stage": stage_name, "nInd": 0, "meanG": np.nan, "varG": np.nan}
    return {
        "stage": stage_name,
        "nInd": int(pop.n_ind),
        "meanG": safeMeanG(pop),
        "varG": safeVarG(pop),
    }

def selectTop(pop, n_keep, varE, reps=1):
    pop = setPheno(pop, varE=varE, reps=reps, simParam=SP)
    n_keep = min(n_keep, pop.n_ind)
    return selectInd(pop, nInd=n_keep, simParam=SP)

def makeProgenyRows(f1_pop, rows_per_cross):
    families = []
    for i in range(min(n_crosses, f1_pop.n_ind)):
        fam = self_pop(f1_pop, nProgeny=rows_per_cross, parents=[i], simParam=SP)
        families.append(fam)
    return mergePops(families)

print("Helper functions defined.")


## Simulate One Breeding Pipeline

This section follows the BRAID workflow:

1. **Elite × elite crossing**
2. **Advance to F2:F3 progeny rows**
3. **Select F2 single plants using early GS-inspired ranking**
4. **Evaluate progeny rows**
5. **Select regional yield test entries**
6. **Advance through two advanced yield test stages**
7. **Create pre-commercial and foundation increase outputs**


In [ ]:
print("Running one pipeline through the BRAID stages...")

# 1. Year 1 crosses
year1_crosses = randCross(elite_parents, nCrosses=n_crosses, nProgeny=f1_per_cross, simParam=SP)

# 2. Advance to F2:F3 progeny rows by selfing
f2f3_progeny_rows = makeProgenyRows(year1_crosses, rows_per_cross=f2f3_per_cross)

# 3. Early single-plant selection (GS-inspired approximation)
# We self each F1 to create a small F2 single-plant candidate set, then select top individuals.
f2_single_candidates = self_pop(year1_crosses, nProgeny=max(1, n_f2_single), simParam=SP)
f2_single_plants = selectTop(f2_single_candidates, n_keep=n_f2_single, varE=varE_early, reps=1)

# 4. Evaluate progeny rows with noisy phenotype
f2f3_progeny_rows = setPheno(f2f3_progeny_rows, varE=varE_early, reps=1, simParam=SP)

# 5. Select regional yield test entries
regional_yield_test = selectInd(f2f3_progeny_rows, nInd=min(n_regional, f2f3_progeny_rows.n_ind), simParam=SP)
regional_yield_test = setPheno(regional_yield_test, varE=varE_regional, reps=rep_regional, simParam=SP)

# 6. Advanced yield test stages
advanced_yield_test_1 = selectInd(regional_yield_test, nInd=min(n_ayt1, regional_yield_test.n_ind), simParam=SP)
advanced_yield_test_1 = setPheno(advanced_yield_test_1, varE=varE_ayt1, reps=rep_ayt1, simParam=SP)

advanced_yield_test_2 = selectInd(advanced_yield_test_1, nInd=min(n_ayt2, advanced_yield_test_1.n_ind), simParam=SP)
advanced_yield_test_2 = setPheno(advanced_yield_test_2, varE=varE_ayt2, reps=rep_ayt2, simParam=SP)

# 7. Downstream outputs
precommercial_test = selectInd(advanced_yield_test_2, nInd=min(n_precommercial, advanced_yield_test_2.n_ind), simParam=SP)
precommercial_test = setPheno(precommercial_test, varE=varE_precommercial, reps=rep_precommercial, simParam=SP)

foundation_increase = selectInd(precommercial_test, nInd=min(n_foundation, precommercial_test.n_ind), simParam=SP)

print("Pipeline complete.")
print("Regional entries:", regional_yield_test.n_ind)
print("AYT1 entries:", advanced_yield_test_1.n_ind)
print("AYT2 entries:", advanced_yield_test_2.n_ind)
print("Precommercial entries:", precommercial_test.n_ind)
print("Foundation entries:", foundation_increase.n_ind)


## Stage Summary for One Pipeline

In [ ]:
stage_rows = [
    stageSummary("elite_parents", elite_parents),
    stageSummary("year1_crosses", year1_crosses),
    stageSummary("f2_single_plants", f2_single_plants),
    stageSummary("f2f3_progeny_rows", f2f3_progeny_rows),
    stageSummary("regional_yield_test", regional_yield_test),
    stageSummary("advanced_yield_test_1", advanced_yield_test_1),
    stageSummary("advanced_yield_test_2", advanced_yield_test_2),
    stageSummary("precommercial_test", precommercial_test),
    stageSummary("foundation_increase", foundation_increase),
]

stage_df = pd.DataFrame(stage_rows)
stage_df


## Multi-Year Recycling of Parents

The BRAID abstraction has a 7-year horizon.  
To reflect repeated annual operation, we recycle the best advanced lines back into the elite parent pool each year.


In [ ]:
records = []

parents = elite_parents

for year in range(1, n_years + 1):
    crosses = randCross(parents, nCrosses=n_crosses, nProgeny=f1_per_cross, simParam=SP)
    rows = makeProgenyRows(crosses, rows_per_cross=f2f3_per_cross)
    rows = setPheno(rows, varE=varE_early, reps=1, simParam=SP)

    ryt = selectInd(rows, nInd=min(n_regional, rows.n_ind), simParam=SP)
    ryt = setPheno(ryt, varE=varE_regional, reps=rep_regional, simParam=SP)

    ayt1 = selectInd(ryt, nInd=min(n_ayt1, ryt.n_ind), simParam=SP)
    ayt1 = setPheno(ayt1, varE=varE_ayt1, reps=rep_ayt1, simParam=SP)

    ayt2 = selectInd(ayt1, nInd=min(n_ayt2, ayt1.n_ind), simParam=SP)
    ayt2 = setPheno(ayt2, varE=varE_ayt2, reps=rep_ayt2, simParam=SP)

    pre = selectInd(ayt2, nInd=min(n_precommercial, ayt2.n_ind), simParam=SP)
    foundation = selectInd(pre, nInd=min(n_foundation, pre.n_ind), simParam=SP)

    records.append({
        "year": year,
        "parent_meanG": safeMeanG(parents),
        "parent_varG": safeVarG(parents),
        "regional_meanG": safeMeanG(ryt),
        "ayt2_meanG": safeMeanG(ayt2),
        "foundation_meanG": safeMeanG(foundation),
        "foundation_nInd": int(foundation.n_ind),
    })

    # Recycle best advanced lines into the parent pool
    merged = mergePops([parents, foundation])
    parents = selectInd(merged, nInd=founder_size, simParam=SP)

results_df = pd.DataFrame(records)
results_df


## Plot Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(results_df["year"], results_df["parent_meanG"], marker="o")
axes[0].set_title("Parent Mean Genetic Value")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Mean G")
axes[0].grid(True, alpha=0.3)

axes[1].plot(results_df["year"], results_df["parent_varG"], marker="o", color="darkorange")
axes[1].set_title("Parent Genetic Variance")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Var G")
axes[1].grid(True, alpha=0.3)

axes[2].plot(results_df["year"], results_df["foundation_meanG"], marker="o", color="forestgreen")
axes[2].set_title("Foundation Output Mean G")
axes[2].set_xlabel("Year")
axes[2].set_ylabel("Mean G")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Summary

This notebook translated the BRAID abstraction into an AlphaSimPy workflow that:

- creates an **elite parent pool**,
- performs **elite × elite crossing**,
- advances material by **selfing**,
- applies **early selection** and **multi-stage testing**,
- produces **pre-commercial** and **foundation increase** outputs,
- and recycles elite outputs into future parent pools across the program horizon.

Because the BRAID abstraction omits some low-level simulation details, the notebook uses explicit, documented assumptions while preserving the intended stage structure and selection flow.
